# Agentic forecasting with skforecast-ai

## What is skforecast-ai?

**skforecast-ai** is a forecasting assistant that pairs a **deterministic forecasting engine** (built on [`skforecast`](https://skforecast.org)) with an **LLM reasoning layer**. Pass it a time series, and it profiles the data, selects a model using established best practices, evaluates it, and returns the forecast, along with the runnable `skforecast` script that produced it.

It is organized around a single core object, the `ForecastingAssistant`, which consists of two complementary components:

+ **Deterministic Engine (Rule-based and Reproducible):** Profiles the data, selects a forecaster and estimator, derives lags and preprocessing steps, runs backtesting, and produces the final forecast. Crucially, it outputs the **exact standalone `skforecast` script** that generated the results. Given the same inputs and configuration, this workflow is guaranteed to be reproducible.

+ **Reasoning Layer (LLM-powered):** Accessed primarily via the `ask()` method, this layer interprets and explains the objects and results you pass to it: data profiles, modeling plans, validation choices, backtesting outputs, and forecasts. The LLM acts strictly as an interpreter; it does not rerun the workflow or silently change modeling recommendations behind the scenes. Agentic features, such as the LLM-guided `refine_plan()` or `create_cv()`, are separate, explicit steps where the LLM suggests adjustments that are then implemented transparently in deterministic code.

## Two ways to use skforecast-ai

**skforecast-ai** supports two distinct workflows using the same underlying forecasting engine:

+ **The Fast Path:** Use this when you want a forecast or backtest result in a single call. The assistant profiles the data, builds the modeling plan, executes the workflow, and returns the results alongside the reproducible `skforecast` code.

+ **The Step-by-Step Path:** Use this when you want granular control to inspect or adjust intermediate decisions. You can manually create a profile, build a plan, optionally refine it with the LLM, define a validation strategy, evaluate the model, and then generate the forecast.

A useful mental model is that forecasting and validation are separate branches. Once you have a `profile` and a `plan`, you can use `forecast()` to produce future predictions directly, or `backtest()` to evaluate the model's performance on historical data.

The `ask()` method is available in both workflows. It can explain a profile, plan, validation setup, backtest result, or answer general forecasting questions, but it will never execute the workflow or modify your parameters without explicit instruction.


<div style="box-sizing:border-box; margin:16px 0; font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif; color:#24292f; max-width:100%;">
  <div style="box-sizing:border-box; display:flex; gap:20px; flex-wrap:wrap; align-items:stretch;">

<!-- Fast path -->
<div style="box-sizing:border-box; flex:1 1 260px; min-width:0; border:1px solid #d0d7de; border-radius:12px; overflow:hidden; display:flex; flex-direction:column;">
    <div style="box-sizing:border-box; background:#0969da; color:#ffffff; padding:12px 16px; font-size:15px; font-weight:700;">Fast path: one call</div>
    <div style="box-sizing:border-box; padding:16px; background:#f6f8fa; flex:1;">
    <p style="margin:0 0 12px 0; font-size:13px;">Profiling, planning and execution happen internally.</p>
    <div style="box-sizing:border-box; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:10px 12px; text-align:center; font-weight:600;">data</div>
    <div style="text-align:center; color:#57606a; font-size:18px; line-height:1.4;">&#8595;</div>
    <div style="box-sizing:border-box; display:flex; gap:12px; flex-wrap:wrap;">
        <div style="box-sizing:border-box; flex:1 1 150px; min-width:0; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:10px;">
        <div style="font-size:11px; color:#57606a; text-transform:uppercase; letter-spacing:.5px; text-align:center; margin-bottom:6px;">Forecast</div>
        <div style="box-sizing:border-box; background:#dbeafe; border:1px solid #0969da; border-radius:8px; padding:8px; text-align:center; font-weight:700;">forecast()<br><span style="font-weight:400; font-size:12px; color:#57606a;">or forecast_code()</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="text-align:center; font-size:12px; color:#24292f;">predictions + code</div>
        </div>
        <div style="box-sizing:border-box; flex:1 1 150px; min-width:0; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:10px;">
        <div style="font-size:11px; color:#57606a; text-transform:uppercase; letter-spacing:.5px; text-align:center; margin-bottom:6px;">Backtesting (validation)</div>
        <div style="box-sizing:border-box; background:#dbeafe; border:1px solid #0969da; border-radius:8px; padding:8px; text-align:center; font-weight:700;">create_cv()<br><span style="font-weight:400; font-size:12px; color:#57606a;">Deterministic, Agentic mode</span><br><span style="font-weight:400; font-size:12px; color:#57606a;">or pass a skforecast TimeSeriesFold object</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="box-sizing:border-box; background:#dbeafe; border:1px solid #0969da; border-radius:8px; padding:8px; text-align:center; font-weight:700;">backtest()<br><span style="font-weight:400; font-size:12px; color:#57606a;">or backtest_code()</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="text-align:center; font-size:12px; color:#24292f;">metrics + predictions + code</div>
        </div>
    </div>
    </div>
</div>

<!-- Step-by-step path -->
<div style="box-sizing:border-box; flex:1.6 1 340px; min-width:0; border:1px solid #d0d7de; border-radius:12px; overflow:hidden; display:flex; flex-direction:column;">
    <div style="box-sizing:border-box; background:#1a7f37; color:#ffffff; padding:12px 16px; font-size:15px; font-weight:700;">Step-by-step path: full control</div>
    <div style="box-sizing:border-box; padding:16px; background:#f6f8fa; flex:1;">
    <p style="margin:0 0 12px 0; font-size:13px;">Build a <code>profile</code> and a <code>plan</code> from your data, then branch into forecasting and backtesting.</p>
    <div style="box-sizing:border-box; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:8px 12px; text-align:center; font-weight:600;">data</div>
    <div style="text-align:center; color:#57606a; font-size:16px; line-height:1.4;">&#8595;</div>
    <div style="box-sizing:border-box; background:#dcfce7; border:1px solid #1a7f37; border-radius:8px; padding:8px 12px; text-align:center; font-weight:700;">profile()</div>
    <div style="text-align:center; color:#57606a; font-size:16px; line-height:1.4;">&#8595;</div>
    <div style="box-sizing:border-box; background:#dcfce7; border:1px solid #1a7f37; border-radius:8px; padding:8px 12px; text-align:center; font-weight:700;">plan()<br><span style="font-weight:400; font-size:12px; color:#57606a;">refine_plan(), optional (Deterministic or Agentic mode)</span></div>
    <div style="text-align:center; color:#57606a; font-size:16px; line-height:1.4;">&#8595;</div>
    <div style="box-sizing:border-box; display:flex; gap:12px; flex-wrap:wrap;">
        <div style="box-sizing:border-box; flex:1 1 150px; min-width:0; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:10px;">
        <div style="font-size:11px; color:#57606a; text-transform:uppercase; letter-spacing:.5px; text-align:center; margin-bottom:6px;">Forecast</div>
        <div style="box-sizing:border-box; background:#dcfce7; border:1px solid #1a7f37; border-radius:8px; padding:8px; text-align:center; font-weight:700;">forecast()<br><span style="font-weight:400; font-size:12px; color:#57606a;">or forecast_code()</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="text-align:center; font-size:12px; color:#24292f;">predictions + code</div>
        </div>
        <div style="box-sizing:border-box; flex:1 1 150px; min-width:0; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:10px;">
        <div style="font-size:11px; color:#57606a; text-transform:uppercase; letter-spacing:.5px; text-align:center; margin-bottom:6px;">Backtesting (validation)</div>
        <div style="box-sizing:border-box; background:#dcfce7; border:1px solid #1a7f37; border-radius:8px; padding:8px; text-align:center; font-weight:700;">create_cv()<br><span style="font-weight:400; font-size:12px; color:#57606a;">Deterministic, Agentic mode</span><br><span style="font-weight:400; font-size:12px; color:#57606a;">or pass a skforecast TimeSeriesFold object</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="box-sizing:border-box; background:#dcfce7; border:1px solid #1a7f37; border-radius:8px; padding:8px; text-align:center; font-weight:700;">backtest()<br><span style="font-weight:400; font-size:12px; color:#57606a;">or backtest_code()</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="text-align:center; font-size:12px; color:#24292f;">metrics + predictions + code</div>
        </div>
    </div>
    </div>
</div>

  </div>

  <!-- ask() banner -->
  <div style="box-sizing:border-box; margin-top:16px; border:1px solid #8250df; border-radius:12px; overflow:hidden;">
    <div style="box-sizing:border-box; background:#8250df; color:#ffffff; padding:10px 16px; font-size:15px; font-weight:700;">LLM reasoning: available at any moment, in any workflow</div>
    <div style="box-sizing:border-box; padding:12px 16px; background:#faf5ff; font-size:13px;">Call <code>ask()</code> before, during or after either path. It can take a <code>profile</code>, a <code>plan</code>, a <code>forecast_result</code>, a <code>backtest_result</code>, or nothing at all (pure Q&amp;A).</div>
  </div>
</div>

The following example walks through the **fast path**: the quickest way to go from raw data to a validated forecast with minimal setup. It is ideal when you want rapid results and trust the assistant to make sensible, baseline modeling decisions on your behalf. If you prefer to understand and control what happens under the hood step by step, visit the [step-by-step path tutorial](./agentic-forecasting-step-by-step.html) for a more detailed walkthrough.


## Assistant initialization

The first step is to instantiate a `ForecastingAssistant`, which will be responsible for executing the entire workflow (profiling, planning, backtesting, and forecasting), as well as explaining the outputs and suggesting improvements.

To activate the optional LLM support, users must pass a string in the format `'provider:model_name'` (for example, `'openai:gpt-5.5'`, `'google:gemini-3-flash-preview'`, `'anthropic:claude-sonnet-5'`, or `'ollama:qwen3:8b'`). For hosted providers, the corresponding API key must be available as an environment variable or passed explicitly when creating the assistant. In this tutorial, we set `send_data_to_llm=False`. This ensures strict data privacy: the LLM receives only metadata and summary statistics, never the raw time series values.

In [1]:
# Data processing
# ==============================================================================
import os
import numpy as np
import pandas as pd
from skforecast.datasets import fetch_dataset

# Plots
# ==============================================================================
from skforecast.plot import set_dark_theme
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
plt.style.use('seaborn-v0_8-darkgrid')

# skforecast and skforecast-ai
# ==============================================================================
import skforecast
import skforecast_ai
from skforecast_ai import ForecastingAssistant
from skforecast.model_selection import TimeSeriesFold

color = '\033[1m\033[38;5;208m' 
print(f"{color}Version skforecast_ai: {skforecast_ai.__version__}")
print(f"{color}Version skforecast: {skforecast.__version__}")

Version skforecast_ai: 0.1.0
Version skforecast: 0.23.0


<div role="note"
     style="background: rgba(0,184,212,.08); border-left: 6px solid #00b8d4; border-radius: 6px; padding: 10px 12px; margin: 1em 0;">
  <p style="display: flex; align-items: center; font-size: 1rem; color: #00b8d4; margin: 0 0 6px 0; font-weight: 600;">
    <span style="margin-right: 6px; font-size: 1.125em;">✏️</span>
    <strong style="font-size: 18px;">Note</strong>
  </p>

  <p>
    If you do not have access to an LLM assistant, you can still follow the full tutorial using only the deterministic methods. Profiling, planning, backtesting, and forecasting all run without an LLM. Only the <code>ask()</code> explanations and the LLM-guided variants of <code>refine_plan()</code> and <code>create_cv()</code> require a configured LLM; their deterministic counterparts (for example, <code>refine_plan()</code> with explicit overrides and <code>prompt=None</code>) work without one.
  </p>
  
</div>

In [2]:
# LLM-enabled assistant
# ==============================================================================
LLM_MODEL = "google:gemini-3-flash-preview"
api_key = os.getenv("GOOGLE_API_KEY")

assistant = ForecastingAssistant(
    llm=LLM_MODEL, api_key=api_key, send_data_to_llm=False
)

# Using aws bedrock
# ==============================================================================
# assistant = ForecastingAssistant(
#     llm='bedrock:eu.anthropic.claude-sonnet-4-6',
#     base_url="eu-west-1"
# )

# Assistant without reasoning layer
# ==============================================================================
# assistant = ForecastingAssistant()

<div role="alert" style="background: rgba(255,145,0,.08); border-left: 6px solid #ff9100; border-radius: 6px; padding: 10px 12px; margin: 1em 0;">

<p style="display:flex; align-items:center; font-size:1rem; color:#ff9100; margin:0 0 6px 0; font-weight:600;">
    <span style="margin-right:6px; font-size:18px;">⚠️</span>
    <strong style="margin-right:6px; font-size:18px;">Your data stays private</strong>
</p>

<p>
By default, enabling an LLM does not send your time-series data to the model provider.
The assistant passes only summary statistics, detected frequency,
seasonality flags and the forecaster configuration, never the raw observations.
To explicitly allow it, pass <code>send_data_to_llm=True</code>.
</p>

</div>

## Data

The data in this document represent the hourly usage of the bike share system in the city of Washington, D.C. during the years 2011 and 2012. In addition to the number of users per hour, information about weather conditions and holidays is available.

In [3]:
# Downloading data
# ==============================================================================
data = fetch_dataset('bike_sharing', raw=True)
data = data[['date_time', 'users', 'holiday', 'weather', 'temp']]
data.head()

╭───────────────────────────────── bike_sharing ──────────────────────────────────╮
│ Description:                                                                    │
│ Hourly usage of the bike share system in the city of Washington D.C. during the │
│ years 2011 and 2012. In addition to the number of users per hour, information   │
│ about weather conditions and holidays is available.                             │
│                                                                                 │
│ Source:                                                                         │
│ Fanaee-T,Hadi. (2013). Bike Sharing Dataset. UCI Machine Learning Repository.   │
│ https://doi.org/10.24432/C5W894.                                                │
│                                                                                 │
│ URL:                                                                            │
│ https://raw.githubusercontent.com/skforecast/skforecast-                        │
│ datasets/main/data/bike_sharing_dataset_clean.csv                               │
│                                                                                 │
│ Shape: 17544 rows x 12 columns                                                  │
╰─────────────────────────────────────────────────────────────────────────────────╯

,date_time,users,holiday,weather,temp
0,2011-01-01 00:00:00,16.0,0.0,clear,9.84
1,2011-01-01 01:00:00,40.0,0.0,clear,9.02
2,2011-01-01 02:00:00,32.0,0.0,clear,9.02
3,2011-01-01 03:00:00,13.0,0.0,clear,9.84
4,2011-01-01 04:00:00,1.0,0.0,clear,9.84


<div role="note"
     style="background: rgba(0,184,212,.08); border-left: 6px solid #00b8d4; border-radius: 6px; padding: 10px 12px; margin: 1em 0;">
  <p style="display: flex; align-items: center; font-size: 1rem; color: #00b8d4; margin: 0 0 6px 0; font-weight: 600;">
    <span style="margin-right: 6px; font-size: 1.125em;">✏️</span>
    <strong style="font-size: 18px;">Note</strong>
  </p>

  <p>
    <code>skforecast-ai</code> is ready to preprocess the data, but it is recommended that users apply their own preprocessing steps before using the assistant. This ensures the data is in the desired format and any necessary transformations have been applied before proceeding with the forecasting workflow.
  </p>
  
</div>

In [4]:
# Interactive plot of time series
# ==============================================================================
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=data['date_time'], y=data['users'], mode='lines', name='Train')
)
fig.update_layout(
    title  = 'Number of users',
    xaxis_title="Time",
    yaxis_title="Users",
    legend_title="Partition:",
    width=800,
    height=400,
    margin=dict(l=20, r=20, t=35, b=20),
    legend=dict(orientation="h", yanchor="top", y=1, xanchor="left", x=0.001)
)
fig.show()

For a deeper walkthrough of the exploratory analysis behind this dataset, see the skforecast example: [Forecasting time series with skforecast, XGBoost, LightGBM and CatBoost](https://cienciadedatos.net/documentos/py39-forecasting-time-series-with-skforecast-xgboost-lightgbm-catboost#Data_exploration).

## Forecasting with the assistant

The `forecast()` method is the fastest way to generate predictions. In a single call, it executes the full pipeline (profile → plan → execute) and returns a `ForecastResult` holding the predictions, the evaluation metrics (when available), and the exact standalone **skforecast** script that produced them.

The behavior is controlled by a single switch, `test_size`, which selects one of two modes:

+ **Prediction mode** (`test_size = None`, the default): The model is trained on the entire dataset and forecasts the next `steps` time points into the future. Because there is no ground truth to compare against, no metrics are returned. If the historical data contains exogenous variables, their future values must be explicitly supplied via the `exog` argument.

+ **Evaluation mode** (`test_size` is set). The dataset is split into train and test sets, the model is trained on the training portion, and predictions for the test window are compared against the held-out actuals to compute metrics. In this mode, the test-set exogenous values are taken from the split, so `exog` must not be passed.


The main arguments are:

| Argument | Type | Description |
|---|---|---|
| `data` | Series, DataFrame, str, Path | Input dataset or path to a CSV file. When a `Series` is passed, the target is taken from its name. |
| `steps` | int | Forecast horizon (number of steps ahead to predict). |
| `target` | str, list of str | Column to forecast. Optional when `data` is a `Series`. Pass a list of column names for wide-format multi-series. |
| `date_column` | str | Column holding the timestamps. When `None`, the index of `data` must already be a `DatetimeIndex`. |
| `exog` | DataFrame | Future exogenous values covering the horizon (at least `steps` rows). Used only in prediction mode, and required there when the data has exogenous variables. |
| `interval` | list of float | Prediction interval as `[lower, upper]` quantiles (e.g. `[0.1, 0.9]` for an 80% interval). `None` disables intervals. |
| `test_size` | int, float, str, Timestamp | Size or start of the test set, selecting the mode above. `int`: last `test_size` observations; `float` in `(0, 1)`: last fraction of observations; `str`/`Timestamp`: first timestamp of the test set. `None` runs prediction mode. |

<div role="note"
    style="background: rgba(0,191,191,.08); border-left: 6px solid #00bfa5;
        border-radius: 6px; padding: 10px 12px; margin: 1em 0;">

<p style="display:flex; align-items:center; font-size:1rem; color:#00bfa5;
        margin:0 0 6px 0; font-weight:600;">
<span style="margin-right:6px; font-size:18px;">💡</span>
<strong style="margin-right:6px; font-size:18px;">Tip</strong>
</p>

<p>

You can also use custom configurations for the <code>forecaster</code>, <code>estimator</code>, <code>estimator_kwargs</code>, <code>lags</code>, and <code>window_features</code> by passing them as arguments to <code>forecast()</code>. This allows you to override the assistant's automatic choices and use your own preferred settings. For example, this code specifies a custom forecaster while still letting the assistant handle the rest of the workflow:

```python
results = assistant.forecast(
              data        = data,
              target      = 'users',
              date_column = "date_time",
              steps       = 36,
              forecaster  = 'ForecasterFoundation'
          )
```

</p>

</div>

The following code demonstrates how to execute the assistant in **evaluation mode**. By explicitly defining the `test_size` parameter (in this case, `test_size=36`), the pipeline automatically reserves the final 36 observations as a test set.

During execution, the assistant profiles the dataset, incorporates the provided exogenous features (holiday, weather, and temp), and trains an appropriate model on the training split. Because the method is triggered in evaluation mode, the resulting `ForecastResult` object provides the out-of-sample predictions, an 80% prediction interval, and the performance metrics evaluated against the held-out ground truth.

In [ ]:
# Execute the forecasting assistant in evaluation mode
# ==============================================================================
results = assistant.forecast(
              data        = data,
              target      = 'users',
              date_column = "date_time",
              steps       = 36,
              interval    = [0.1, 0.9],  # 80% prediction interval
              test_size   = 36           # Last 36 hours as test set (evaluation mode)
          )

In [6]:
# Inspect the performance metrics and the resulting predictions
# ==============================================================================
display(results.metrics)
display(results.predictions.head())

,series,MAE,MSE,MASE,MAPE
0,users,32.013374,2758.8615,0.497185,0.415688


,pred,lower_bound,upper_bound
2012-12-30 12:00:00,146.512223,111.097550,181.185404
2012-12-30 13:00:00,138.768717,96.950185,178.804105
2012-12-30 14:00:00,145.133323,92.701496,185.332563
2012-12-30 15:00:00,140.891323,89.525337,185.272888
2012-12-30 16:00:00,139.538959,86.994814,180.904345


The `show_code()` method allows users to retrieve the exact code used to generate the forecast.

In [7]:
# skforecast code that generated the results
# ==============================================================================
results.show_code()

╭────────────────────────────────────── Generated code ───────────────────────────────────────╮
│ import pandas as pd                                                                         │
│ from sklearn.metrics import mean_absolute_error, mean_squared_error,                        │
│ mean_absolute_percentage_error                                                              │
│ from skforecast.metrics import mean_absolute_scaled_error                                   │
│ from lightgbm import LGBMRegressor                                                          │
│ from skforecast.preprocessing import RollingFeatures, CalendarFeatures                      │
│ from skforecast.recursive import ForecasterRecursive                                        │
│                                                                                             │
│ # Load data                                                                                 │
│ data = pd.read_csv('data.csv')                                                              │
│                                                                                             │
│ data['date_time'] = pd.to_datetime(data['date_time'])                                       │
│ data = data.set_index('date_time')                                                          │
│ data = data.asfreq('h')                                                                     │
│ data = data.sort_index()                                                                    │
│                                                                                             │
│ # Train/test split                                                                          │
│ end_train = '2012-12-30 11:00:00'  # last training date, adjust to change the split point   │
│ data_train = data.loc[:end_train]                                                           │
│ data_test  = data.loc[data.index > end_train]                                               │
│ exog_features = ['holiday', 'weather', 'temp']                                              │
│                                                                                             │
│ print(                                                                                      │
│     f"Train dates : {data_train.index.min()} --- {data_train.index.max()}                   │
│ (n={len(data_train)})"                                                                      │
│ )                                                                                           │
│ print(                                                                                      │
│     f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}                     │
│ (n={len(data_test)})"                                                                       │
│ )                                                                                           │
│                                                                                             │
│ window_features = RollingFeatures(                                                          │
│     stats        = ['mean', 'std', 'mean', 'mean'],                                         │
│     window_sizes = [3, 3, [38;2;174;129;255;

To generate forecasts for true future periods (starting immediately after the last observation in the dataset), the assistant is executed in **prediction mode**. This mode is active by default when `test_size=None`. In this configuration, the pipeline trains the optimal model using the entire available dataset and projects the predictions forward by the specified number of `steps`. Because there is no holdout data to serve as ground truth, the resulting `ForecastResult` object does not compute performance metrics. Additionally, if the historical data relies on exogenous features, a DataFrame containing their future values spanning the entire forecast horizon must be explicitly supplied via the `exog` parameter.

In [8]:
# Forecast the next 36 hours using the entire dataset (prediction mode)
# ==============================================================================
# Simulate future values of exogenous variables for the next 36 hours
exog = data[['holiday', 'weather', 'temp']].tail(36).copy()
exog.index = pd.date_range(
    start=pd.to_datetime(data['date_time'].max()) + pd.Timedelta(hours=1), periods=36, freq='h'
)

results_pred = assistant.forecast(
                   data        = data,
                   target      = 'users',
                   date_column = "date_time",
                   steps       = 36,
                   interval    = [0.1, 0.9],  # 80% prediction interval
                   test_size   = None,        # Use the entire dataset for training (prediction mode)
                   exog        = exog         # Future values of exogenous variables for the next 36 hours
               )

display(results_pred.predictions.head())

,pred,lower_bound,upper_bound
2013-01-01 00:00:00,27.442927,11.127479,47.557452
2013-01-01 01:00:00,13.318240,3.310496,27.242435
2013-01-01 02:00:00,9.208553,2.380079,19.120340
2013-01-01 03:00:00,5.329501,0.570137,10.013675
2013-01-01 04:00:00,5.979160,1.305255,10.265962


Both modes return a `ForecastResult`, a lightweight container that bundles everything the assistant used and produced, so you can inspect the outputs, audit the decisions, or lift the code straight into production.

| Attribute | Type | Description |
|---|---|---|
| `predictions` | DataFrame | Forecasted values for the requested steps. When intervals (or quantiles) are requested, the bound columns are included alongside the point predictions. |
| `metrics` | DataFrame, None | Evaluation metrics (`MAE`, `MSE`, `MASE`), one row per series. `None` in prediction mode, where there is no ground truth to score against. |
| `code` | str | The exact standalone **skforecast** script that produced the forecast, deterministic and ready to run on its own. |
| `profile` | `ForecastingProfile` | The data profile behind the forecast: metadata, summary statistics, detected frequency and seasonality, and the high-level modeling decisions. |
| `plan` | `ForecastPlan` | The detailed configuration that was executed: forecaster, estimator, lags, window features, preprocessing, and interval settings. |

Displaying the object in a notebook renders a rich summary of all of the above; the raw script is also available through `results.show_code()`.

In [9]:
# Full results object
# ==============================================================================
results_pred

              Dataset Profile              
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property       ┃ Value                  ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Format         │ single                 │
├────────────────┼────────────────────────┤
│ Series         │ 1                      │
├────────────────┼────────────────────────┤
│ Observations   │ 17544                  │
├────────────────┼────────────────────────┤
│ Frequency      │ h                      │
├────────────────┼────────────────────────┤
│ Target         │ users                  │
├────────────────┼────────────────────────┤
│ Exog columns   │ holiday, weather, temp │
├────────────────┼────────────────────────┤
│ Missing target │ none                   │
└────────────────┴────────────────────────┘

                                        Recommendation                                         
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property              ┃ Value                                                               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Task type             │ single_series                                                       │
├───────────────────────┼─────────────────────────────────────────────────────────────────────┤
│ Forecaster            │ ForecasterRecursive                                                 │
├───────────────────────┼─────────────────────────────────────────────────────────────────────┤
│ Forecaster candidates │ ForecasterRecursive, ForecasterDirect, ForecasterFoundation,        │
│                       │ ForecasterStats                                                     │
├───────────────────────┼─────────────────────────────────────────────────────────────────────┤
│ Estimator             │ LGBMRegressor                                                       │
├───────────────────────┼─────────────────────────────────────────────────────────────────────┤
│ Estimator candidates  │ LGBMRegressor, XGBRegressor, Ridge                                  │
└───────────────────────┴─────────────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────── Explanation ────────────────────────────────────────╮
│                                                                                             │
│  A single-series ML forecaster (ForecasterRecursive) is recommended. Data: 17544            │
│  observations, 'h' frequency. Alternative forecasters: ['ForecasterDirect',                 │
│  'ForecasterFoundation', 'ForecasterStats']. Estimator: LGBMRegressor. A gradient boosting  │
│  model is preferred for a dataset of this size (17544 observations). Alternative            │
│  estimators: ['XGBRegressor', 'Ridge']. 3 exogenous variables (1 categorical) available as  │
│  predictors.                                                                                │
│                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────╯
                                         Forecast Plan                                         
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property          ┃ Value                                                                   ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Forecaster        │ ForecasterRecursive                                                     │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Estimator         │ LGBMRegressor                                                           │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Steps             │ 36              

## Interpreting results

After executing the workflow, you can leverage the LLM reasoning layer to interpret the outputs. By passing the `ForecastResult` object to the `ask()` method, the assistant analyzes the generated predictions, performance metrics, and model configuration. This allows you to quickly extract actionable insights (such as contextualizing specific error metrics, identifying predicted trends, or diagnosing model behavior) directly in natural language.

In [10]:
# Asking the assistant about the forecast results
# ==============================================================================
explanation = assistant.ask(
    prompt = "Explain the results of the forecast, including the metrics and predictions.",
    forecast_result = results
)

In [11]:
# Show the explanation generated by the LLM
# ==============================================================================
explanation.show_explanation()

╭──────────────────────────────────────── Explanation ────────────────────────────────────────╮
│                                                                                             │
│  Based on the pre-computed results, here is an explanation of the forecast performance and  │
│  the predicted values.                                                                      │
│                                                                                             │
│  Model Performance and Metrics                                                              │
│                                                                                             │
│  The model was evaluated using several key metrics to determine its accuracy relative to    │
│  the data's scale and a simple baseline.                                                    │
│                                                                                             │
│   • MASE (Mean Absolute Scaled Error): 0.497 This is the most important metric for          │
│     validation. A MASE below 1.0 indicates that the model is performing better than a       │
│     naive baseline (e.g., predicting that "next hour will be the same as the previous       │
│     period"). At 0.497, your model is approximately twice as accurate as a naive forecast,  │
│     suggesting it has successfully captured the underlying hourly and daily patterns.       │
│   • MAE (Mean Absolute Error): 32.01 On average, the model's predictions deviate from the   │
│     actual number of users by approximately 32 users. This provides a clear sense of the    │
│     error magnitude in the same units as your target variable.                              │
│   • MAPE (Mean Absolute Percentage Error): 0.416 (41.6%) This represents the average        │
│     relative error. While 41% may appear high, percentage errors are often inflated in      │
│     time series with low values or high volatility, which is common in hourly user data.    │
│   • MSE (Mean Squared Error): 2758.86 This metric penalizes larger errors more heavily      │
│     than smaller ones. It is useful for identifying if the model is making occasional       │
│     "large" misses.                                                                         │
│                                                                                             │
│  Prediction Summary and Uncertainty                                                         │
│                                                                                             │
│  The forecast covers a 36-hour horizon, starting from December 30, 2012.                    │
│                                                                                             │
│   • Point Forecasts: The model predicts a wide range of activity, with values as low as     │
│     5.26 users and peaking at 260.8 users. The average predicted volume across the horizon  │
│     is 103.2 users.                                                                         │
│   • Trend Observation: Looking at the end of the period (late night on December 31st), the  │
│     model predicts a significant drop in users (down to ~40 users by 23:00), which likely   │
│     aligns with expected late-night/early-morning behavior.                                 │
│   • 80% Prediction Intervals: The "lower_bound" and "upper_bound" columns define the range  │
│     where the actual value is expected to fall 80% of the time.                             │
│      • For the first prediction (12:00), the model expects 146 users, but given the         │
│        uncertainty, the count could realistically range between 111 and 181.                │
│      • As the forecast moves further into the future (e.g., Dec 31, 19:00), the interval    │
│        widens (99 to 201), reflecting the natural increase in uncertainty as the model      │
│        predicts further away from the known data.             

## Refining the modeling plan

Beyond explaining results, the assistant can also help you to improve the modeling strategy using the `refine_plan()` method. This method accepts an existing `profile` and `plan` and returns an updated `ForecastPlan`. It operates in two distinct modes:

- **Deterministic mode (Explicit Overrides)**: When no prompt is provided, you can pass explicit configuration overrides (such as `forecaster`, `estimator`, `estimator_kwargs`, `steps`, `interval`, `lags`, or `window_features`). Only the explicitly specified fields are updated; the remaining configuration is deterministically re-derived from the original plan.

- **LLM mode (Domain Knowledge Integration)**: When a prompt is provided, the assistant leverages the LLM to interpret your natural-language domain knowledge. The agent suggests domain-specific `lags` and `window_features`, which are then merged into the plan. To maintain strict interpretability, the LLM’s reasoning is appended to the returned plan's `explanation` attribute, allowing you to trace exactly why each feature was proposed.

The following example demonstrates the LLM mode. By supplying a prompt with context about the dataset, we instruct the assistant to engineer more appropriate lags and window features. We then execute `forecast()` using the refined plan and compare the new metrics against our original baseline run.

Keep in mind that a refined plan is a **hypothesis, not a guaranteed improvement**. The LLM proposes lags and window features from the domain knowledge you provide, but only a proper evaluation can confirm whether they actually help. Always compare the refined plan against the original before adopting it, and check that the suggested features are sensible for your data and use case rather than trusting them blindly.

In [12]:
# Ask the assistant to improve the forecast accuracy
# ==============================================================================
prompt  = (
    "I'm forecasting hourly bike rentals. Demand follows a clear daily rhythm with "
    "rush-hour peaks, and it changes between weekdays and weekends. It's also usually "
    "similar to what happened at the same time last week, and the last few hours give "
    "a good sense of the current trend. Please pick lags and rolling features that fit this."
)

refined_plan = assistant.refine_plan(
                   profile = results_pred.profile,
                   plan    = results_pred.plan,
                   prompt  = prompt
               )

In [13]:
# Refined plan proposed by the assistant
# ==============================================================================
refined_plan

                                         Forecast Plan                                         
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property          ┃ Value                                                                   ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Forecaster        │ ForecasterRecursive                                                     │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Estimator         │ LGBMRegressor                                                           │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Steps             │ 36                                                                      │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Frequency         │ h                                                                       │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Lags              │ [1, 2, 3, 23, 24, 25, 167, 168, 169]  (LLM-suggested)                   │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Window features   │ [{'stats': ['mean'], 'window_size': 3}, {'stats': ['mean', 'std'],      │
│                   │ 'window_size': 24}, {'stats': ['mean'], 'window_size': 168}]            │
│                   │ (LLM-suggested)                                                         │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Calendar features │ ['hour', 'day_of_week', 'weekend', 'month'] (raw ordinal encoding)      │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Use exog          │ True                                                                    │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Interval          │ [0.1, 0.9]                                                              │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Interval method   │ bootstrapping                                                           │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Primary metric    │ mean_absolute_error                                                     │
├───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Preprocessing     │   - handle_categorical_exog: Categorical exogenous variables detected:  │
│                   │ ['weather']. These are handled automatically by skforecast              │
│                   │ (categorical_features='auto').                                          │
└───────────────────┴─────────────────────────────────────────────────────────────────────────┘

╭───────────────────────────────────── Plan Explanation ──────────────────────────────────────╮
│                                                                                             │
│  Plan: ForecasterRecursive + LGBMRegressor. Lags: [1, 2, 3, 23, 24, 25, 167, 168, 169].     │
│  Window features: ['mean(window=3)', 'mean(window=24)', 'std(window=24)',                   │
│  'mean(window=168)']. Calendar features: ['hour', 'day_of_week', 'weekend', 'month'] (raw   │
│  ordinal encoding). Prediction intervals via bootstrapping. NaN rows kept (NaN-tolerant     │
│  estimator). Exogenous variables included. MAE is interpretable, robust to outliers, and    │
│  works at any scale.                                                                        │
│                                                                                             │
│  LLM Refinement Reasoning: To capture the daily rhythm and ru

In [14]:
# Run the forecast with the refined plan 
# ==============================================================================
results_refined = assistant.forecast(
                      data        = data,
                      target      = 'users',
                      date_column = "date_time",
                      steps       = 36,
                      test_size   = 36,           # Last 36 hours as test set (evaluation mode)
                      plan        = refined_plan  # Refined plan proposed by the assistant
                  )

In [15]:
# Results of the forecast with the refined plan
# ==============================================================================
results_refined.metrics

,series,MAE,MSE,MASE,MAPE
0,users,53.633011,5982.969529,0.83295,0.57889


In [16]:
# Results of the forecast with the original plan
# ==============================================================================
results.metrics

,series,MAE,MSE,MASE,MAPE
0,users,32.013374,2758.8615,0.497185,0.415688


As noted above, a refined plan is a hypothesis, not a guaranteed improvement. The LLM may propose lags or window features that are not relevant to the series, or it may misread the domain knowledge you provided, so the refined plan can perform worse than the original.

More importantly, here we evaluate the change on a single 36-hour test window, which is far too small to draw a reliable conclusion. A single split can favor either plan by chance. To decide whether the refinement genuinely helps, evaluate both plans with a backtest over multiple folds (see the next section), which averages performance across many test windows and gives a far more trustworthy comparison.

## Backtesting

In time series forecasting, backtesting is the process of evaluating a predictive model by retrospectively simulating its performance on historical data. It functions as a specialized form of temporal cross-validation, ensuring that data leakage is prevented by strictly respecting the chronological order of observations.

The reliability of backtesting metrics depends entirely on how closely the evaluation setup mirrors production conditions. To obtain trustworthy performance estimates, the backtesting strategy must accurately reflect the real-world prediction horizon, refit frequency, and data availability.

In **skforecast-ai**, this validation strategy is governed by the `TimeSeriesFold` object. It defines exactly how the historical data is partitioned into successive training and test windows. It controls critical parameters such as the forecast horizon (`steps`), the starting point for evaluation (`initial_train_size`), and whether the model is periodically retrained as the window rolls forward (`refit`). Because these choices dictate the interpretation of the resulting metrics, correctly configuring the `TimeSeriesFold` is the most critical decision in the evaluation phase. For a comprehensive overview of these mechanics, refer to the skforecast [Backtesting user guide](https://skforecast.org/latest/user_guides/backtesting).

skforecast-ai provides three distinct ways to define this validation strategy, ranging from full manual control to automated generation:

+ **Explicit Instantiation (Recommended)**: Manually construct a `TimeSeriesFold` and pass it directly to the `backtest()` method. If you already know the exact operational constraints of your production environment, this is the most explicit and reproducible approach, keeping the validation setup entirely under your control.

+ **LLM Mode (`create_cv()` with a prompt)**: Describe your operational use case in natural language (e.g., required horizon, retraining frequency, trusted historical depth). The LLM reasoning layer interprets this context and translates it into a strictly defined `TimeSeriesFold`, materializing the setup into deterministic code.

+ **Deterministic Mode (`create_cv()` without a prompt)**: Allow the assistant to automatically derive a sensible `TimeSeriesFold` from the existing profile and plan using rule-based defaults. You can still explicitly override individual parameters such as `initial_train_size` or `refit` as needed.

The following example demonstrates the three approaches.

In [17]:
# Explicit Instantiation of the cv
# ==============================================================================
# Create your own TimeSeriesFold object
end_train = '2012-08-31 23:59:00'
cv = TimeSeriesFold(
         steps              = 36, 
         initial_train_size = end_train, 
         refit              = False,
         verbose            = False
     )

results_backtest = assistant.backtest(
                       data        = data,
                       target      = 'users',
                       date_column = "date_time",
                       cv          = cv,          # TimeSeriesFold object
                       interval    = [0.1, 0.9],  # 80% prediction interval
                   )

  0%|          | 0/82 [00:00<?, ?it/s]

In [18]:
# Show backtest results
# ==============================================================================
results_backtest.show_explanation()
display(results_backtest.metrics)
display(results_backtest.predictions.head())

╭──────────────────────────────────────── Explanation ────────────────────────────────────────╮
│                                                                                             │
│  Initial training up to 2012-08-31 23:59:00, fixed window, no refit, 36-step horizon, 82    │
│  folds. Results — mean_absolute_error: 46.3122, mean_squared_error: 5495.8816,              │
│  mean_absolute_scaled_error: 0.7496, mean_absolute_percentage_error: 0.4727.                │
│                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────╯

,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,46.312218,5495.881638,0.749598,0.472735


,fold,pred,lower_bound,upper_bound
2012-09-01 00:00:00,0,124.163668,99.669766,149.384335
2012-09-01 01:00:00,0,96.989454,68.523139,127.651105
2012-09-01 02:00:00,0,65.894182,32.969786,97.883129
2012-09-01 03:00:00,0,29.961028,10.347714,60.808462
2012-09-01 04:00:00,0,9.103018,2.980393,28.085362


After running the workflow, you can ask the assistant to explain the backtesting results. The `ask()` method can take a `BacktestResult` object and provide insights about the strategy, the metrics, or any other relevant information.

In [19]:
# Asking the assistant about the backtesting
# ==============================================================================
explanation = assistant.ask(
    prompt = "Explain the results of the backtesting, including the strategy, metrics, and predictions.",
    backtest_result = results_backtest
)

In [20]:
# Show the explanation generated by the LLM
# ==============================================================================
explanation.show_explanation()

╭──────────────────────────────────────── Explanation ────────────────────────────────────────╮
│                                                                                             │
│  The following sections explain the backtesting strategy and evaluation results for your    │
│  forecasting model.                                                                         │
│                                                                                             │
│  Backtesting Strategy                                                                       │
│                                                                                             │
│  The model was evaluated using a fixed-train backtesting strategy. This approach provides   │
│  a realistic simulation of how the model performs over time without the computational cost  │
│  of frequent retraining.                                                                    │
│                                                                                             │
│   • Training and Refit: The model was trained once using data up to 2012-08-31. Because     │
│     refit was set to False, this single model was used to generate forecasts for the        │
│     remainder of the dataset.                                                               │
│   • Horizon and Folds: Predictions were made in blocks of 36 hours (the forecast horizon).  │
│     A total of 82 folds (fold 0 to 81) were processed, moving forward 36 steps at a time    │
│     (fold_stride=36) to cover the entire evaluation period.                                 │
│   • Data Preparation: The strategy accounts for the hourly frequency of your data and       │
│     includes exogenous variables (holiday, weather, temp). The model also automatically     │
│     generated calendar features like the hour of the day and day of the week to capture     │
│     daily and weekly seasonality.                                                           │
│                                                                                             │
│  Metric Interpretation                                                                      │
│                                                                                             │
│  The performance metrics indicate that the model is effectively capturing the underlying    │
│  patterns of the "users" series.                                                            │
│                                                                                             │
│   • Mean Absolute Scaled Error (MASE): 0.75                                                 │
│     This is the most critical metric for assessing value. A MASE of 0.75 (which is less     │
│     than 1.0) means the model is 25% more accurate than a "naive" forecast (which simply    │
│     predicts the value from the previous period). This indicates the complex lags and       │
│     window features are significantly improving prediction quality.                         │
│   • Mean Absolute Error (MAE): 46.31                                                        │
│     On average, the model's predictions deviate from the actual number of users by          │
│     approximately 46.3 units. Given that the average predicted value is 236, this error     │
│     level represents a reasonable margin for many operational use cases.                    │
│   • Mean Absolute Percentage Error (MAPE): 47.27%                                           │
│     While this percentage seems high, MAPE can be sensitive to low volume periods (e.g.,    │
│     late-night hours where "users" may be near zero). The MASE and MAE provide a more       │
│     robust view of performance for this specific dataset.                                   │
│                                                                                             │
│  Predictions and Uncertainty                                  

Rather than manually configuring `TimeSeriesFold` parameters, you can describe your target backtesting strategy in natural language and allow the assistant's LLM layer to translate it into a rigorous cross-validation schema. Once you have generated a data `profile` and a modeling `plan`, simply pass a prompt detailing your operational constraints, such as the required forecast horizon, historical training depth, refitting frequency, or expected data gaps. The assistant processes this context and returns a fully configured `TimeSeriesFold` object, accompanied by an explicit explanation of its design choices. This transparency ensures you can thoroughly audit and verify the validation strategy before executing the backtest.

In [21]:
# Let the assistant create the TimeSeriesFold for you
# ==============================================================================
# Profile and create a plan for your data
profile = assistant.profile(data, target="users", date_column="date_time")
plan = assistant.plan(profile, steps=36)

# Call create_cv with a prompt that describes your prediction strategy
prompt = (
    "I forecast bike demand 36 hours ahead. "
    "The model should be trained once on all data up to the end of August 2012, 23:59. "
    "Do not refit the model as the window rolls forward."
)
cv, cv_explanation = assistant.create_cv(
                         profile = profile,
                         plan    = plan,
                         prompt  = prompt
                     )

In [22]:
# TimeSeriesFold object
# ==============================================================================
cv

TimeSeriesFold(
    initial_train_size    = 2012-08-31 23:59:59,
    steps                 = 36,
    fold_stride           = 36,
    window_size           = None,
    differentiation       = None,
    refit                 = False,
    fixed_train_size      = False,
    gap                   = 0,
    skip_folds            = None,
    allow_incomplete_fold = True,
    return_all_indexes    = False,
    verbose               = False,
)

In [23]:
# LLM Reasoning
# ==============================================================================
import textwrap
print(textwrap.fill(cv_explanation, width=88))

The initial training size is set to the specific cutoff date requested (end of August
2012). Since you explicitly requested that the model not be refit as the window rolls
forward, 'refit' is set to False. This configuration will train the model once on the
data until August 31st and then evaluate its performance on successive 36-hour test
windows (steps). Initial training up to 2012-08-31 23:59:59, expanding window, no refit,
36-step horizon, 82 folds.


Since the `prompt` correctly describes the intended use case, the `cv` object returned by `create_cv()` is the same as the one we built manually above. However it was derived from a natural-language description rather than explicit parameters. The assistant also returns a `cv_explanation` string that details the choices it made. This allows you to verify that the resulting `TimeSeriesFold` matches your intended strategy.

The backtesting workflow returns a `BacktestResult`, a lightweight container that bundles everything the assistant used and produced, so you can inspect the outputs, audit the decisions, or lift the code straight into production.

| Attribute | Type | Description |
|---|---|---|
| `predictions` | DataFrame | Full out-of-sample backtest predictions across all folds. When intervals (or quantiles) are requested, the bound columns are included alongside the point predictions. |
| `metrics` | DataFrame | Backtesting metrics (`MAE`, `MSE`, `MASE`) returned by **skforecast**, one row per series, computed over the reserved test window. |
| `cv_config` | dict | The resolved `TimeSeriesFold` parameters (steps, initial train size, refit, gap, etc.), kept for full traceability of the validation strategy. |
| `code` | str | The exact standalone **skforecast** script that reproduces the backtesting workflow, deterministic and ready to run on its own. |
| `explanation` | str | A human-readable summary of the backtesting configuration and what the results mean. |
| `profile` | `ForecastingProfile` | The data profile behind the backtest: metadata, summary statistics, detected frequency and seasonality, and the high-level modeling decisions. |
| `plan` | `ForecastPlan` | The detailed configuration that was executed: forecaster, estimator, lags, window features, preprocessing, and interval settings. |

Displaying the object renders a rich summary of all of the above; the raw script is also available through `results_backtest.show_code()`.

In [24]:
# Full results object
# ==============================================================================
results_backtest

╭──────────────────────────────────────── Explanation ────────────────────────────────────────╮
│                                                                                             │
│  Initial training up to 2012-08-31 23:59:00, fixed window, no refit, 36-step horizon, 82    │
│  folds. Results — mean_absolute_error: 46.3122, mean_squared_error: 5495.8816,              │
│  mean_absolute_scaled_error: 0.7496, mean_absolute_percentage_error: 0.4727.                │
│                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────╯
       Cross-Validation Configuration       
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Parameter          ┃               Value ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ steps              │                  36 │
├────────────────────┼─────────────────────┤
│ initial_train_size │ 2012-08-31 23:59:00 │
├────────────────────┼─────────────────────┤
│ refit              │               False │
├────────────────────┼─────────────────────┤
│ fixed_train_size   │                True │
├────────────────────┼─────────────────────┤
│ gap                │                   0 │
├────────────────────┼─────────────────────┤
│ fold_stride        │                  36 │
├────────────────────┼─────────────────────┤
│ differentiation    │                None │
└────────────────────┴─────────────────────┘
                                       Backtest Metrics                                        
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ mean_absolute_error ┃ mean_squared_error ┃ mean_absolute_scaled_… ┃ mean_absolute_percenta… ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│             46.3122 │          5495.8816 │                 0.7496 │                  0.4727 │
└─────────────────────┴────────────────────┴────────────────────────┴─────────────────────────┘
                    Backtest Predictions (2928 rows)                    
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Index               ┃    fold ┃     pred ┃ lower_bound ┃ upper_bound ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ 2012-09-01 00:00:00 │  0.0000 │ 124.1637 │     99.6698 │    149.3843 │
├─────────────────────┼─────────┼──────────┼─────────────┼─────────────┤
│ 2012-09-01 01:00:00 │  0.0000 │  96.9895 │     68.5231 │    127.6511 │
├─────────────────────┼─────────┼──────────┼─────────────┼─────────────┤
│ 2012-09-01 02:00:00 │  0.0000 │  65.8942 │     32.9698 │     97.8831 │
├─────────────────────┼─────────┼──────────┼─────────────┼─────────────┤
│ 2012-09-01 03:00:00 │  0.0000 │  29.9610 │     10.3477 │     60.8085 │
├─────────────────────┼─────────┼──────────┼─────────────┼─────────────┤
│ 2012-09-01 04:00:00 │  0.0000 │   9.1030 │      2.9804 │     28.0854 │
├─────────────────────┼─────────┼──────────┼─────────────┼─────────────┤
│ ...                 │     ... │      ... │         ... │         ... │
├─────────────────────┼─────────┼──────────┼─────────────┼─────────────┤
│ 2012-12-31 19:00:00 │ 81.0000 │ 145.8848 │     97.1293 │    180.5958 │
├─────────────────────┼─────────┼──────────┼─────────────┼─────────────┤
│ 2012-12-31 20:00:00 │ 81.0000 │ 103.3326 │     67.3546 │    140.2631 │
├─────────────────────┼─────────┼──────────┼─────────────┼─────────────┤
│ 2012-12-31 21:00:00 │ 81.0000 │  64.1944 │     33.2954 │     86.5947 │
├─────────────────────┼─────────┼──────────┼─────────────┼─────────────┤
│ 2012-12-31 22:00:00 │ 81.0000 │  46.9473 │     23.0095 │     64.8209 │
├─────────────────────┼─────────┼──────────┼─────────────┼─────────────┤
│ 2012-12-31 23:00:00 │ 81.0000 │  30.5361 │     12.3134 │     45.8523 │
└─────────────────────┴─────────┴──────────┴─────────────┴─────────────┘
              Dataset Profile              


This tutorial covered the **fast path** of `skforecast-ai`: the most efficient route from raw time series data to a validated forecast with minimal manual configuration. This approach is highly effective for rapidly generating robust baseline models when you trust the assistant to implement sensible default strategies.

However, if your use case requires granular control to inspect, audit, or modify the intermediate objects (such as the data profile, the modeling plan, or the validation schema) before executing the next phase, we recommend exploring the [step-by-step workflow](./agentic-forecasting-step-by-step.html) for a comprehensive, stage-by-stage walkthrough of the underlying architecture.